# Nettoyage du Dataset OLAP — Phase 1

**Objectif :** Préparer le dataset enrichi (`olap_dataset_final.xlsx`) avant chargement dans PostgreSQL.

| Étape | Description |
|---|---|
| **1** | Catégorisation des lignes (ACTIF / INACTIF / DONNEE_MANQUANTE) |
| **2** | Marquage des outliers physiques (méthode IQR) |
| **3** | Correction de la divergence géographique |
| **4** | Détection et suppression des doublons |
| **5** | Validation finale et rapport de qualité |

**Entrée :** `olap_dataset_final.xlsx`  
**Sortie :** `olap_clean.xlsx`

## 0. Imports et Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)

FEATURES = ['olt_power_tx', 'olt_rx_ont_power', 'onu_power_tx', 'onu_power_rx']

INPUT_FILE  = 'olap_dataset_final.xlsx'
OUTPUT_FILE = 'olap_clean.xlsx'

print('Libraries chargees avec succes')

Libraries chargees avec succes


## Chargement des données sources

In [2]:
df        = pd.read_excel(INPUT_FILE, sheet_name='FAIT_MESURE_OLT')
dim_equip = pd.read_excel(INPUT_FILE, sheet_name='DIM_EQUIPEMENT')
dim_geo   = pd.read_excel(INPUT_FILE, sheet_name='DIM_GEOGRAPHIE')
dim_temps = pd.read_excel(INPUT_FILE, sheet_name='DIM_TEMPS')

print(f'FAIT_MESURE_OLT : {df.shape[0]} lignes x {df.shape[1]} colonnes')
print(f'DIM_EQUIPEMENT  : {dim_equip.shape[0]} enregistrements')
print(f'DIM_GEOGRAPHIE  : {dim_geo.shape[0]} enregistrements')
print(f'DIM_TEMPS       : {dim_temps.shape[0]} enregistrements')

# Forcer la conversion numerique des colonnes de mesures
for col in FEATURES:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

nb_initial = len(df)
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'olap_dataset_final.xlsx'

---
## Étape 1 — Catégorisation des lignes

**Même logique que le notebook de détection :**
- `INACTIF` : port désactivé (`ADMIN=0`) ou ligne hors service (`OPER=0`)
- `DONNEE_MANQUANTE` : ligne active mais mesures absentes (anomalie de collecte)
- `ACTIF` : ligne active avec toutes les mesures disponibles

In [ ]:
def categoriser_ligne(row):
    mesures_nulles = pd.isna(row[FEATURES]).any()
    if row['admin_status'] == 0 or row['oper_status'] == 0:
        return 'INACTIF'
    elif mesures_nulles:
        return 'DONNEE_MANQUANTE'
    else:
        return 'ACTIF'

df['statut_ligne'] = df.apply(categoriser_ligne, axis=1)

counts = df['statut_ligne'].value_counts()
print('Distribution des statuts :')
for statut, n in counts.items():
    pct = 100 * n / len(df)
    print(f'  {statut:<22} : {n:>5} lignes ({pct:.1f}%)')

In [ ]:
# Visualisation
colors = {'ACTIF': '#2ecc71', 'INACTIF': '#e74c3c', 'DONNEE_MANQUANTE': '#f39c12'}
counts.plot(
    kind='bar',
    color=[colors.get(s, '#95a5a6') for s in counts.index],
    edgecolor='white',
    width=0.6
)
plt.title('Distribution des statuts de ligne', fontsize=14, fontweight='bold')
plt.xlabel('Statut')
plt.ylabel('Nombre de lignes')
plt.xticks(rotation=0)
for i, v in enumerate(counts):
    plt.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Étape 2 — Marquage des outliers physiques (Méthode IQR)

**Principe :** On ne supprime pas les outliers ! Dans le contexte XPON, un outlier est souvent une vraie anomalie réseau. On les **marque** avec `is_outlier = 1` pour l'analyse ML.

**Plages physiques XPON acceptables :**

| Mesure | Min | Max |
|---|---|---|
| OLT_POWER_TX | 0.5 dBm | 5.0 dBm |
| OLT_RX_ONT_POWER | -30.0 dBm | -8.0 dBm |
| ONU_POWER_TX | 0.5 dBm | 5.0 dBm |
| ONU_POWER_RX | -30.0 dBm | -8.0 dBm |

In [ ]:
PHYSICAL_LIMITS = {
    'olt_power_tx':     (0.5,  5.0),
    'olt_rx_ont_power': (-30.0, -8.0),
    'onu_power_tx':     (0.5,  5.0),
    'onu_power_rx':     (-30.0, -8.0),
}

df_actif = df[df['statut_ligne'] == 'ACTIF'].copy()
df['is_outlier'] = 0
outlier_summary = {}

for col in FEATURES:
    Q1 = df_actif[col].quantile(0.25)
    Q3 = df_actif[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = max(Q1 - 3 * IQR, PHYSICAL_LIMITS[col][0])
    upper = min(Q3 + 3 * IQR, PHYSICAL_LIMITS[col][1])

    outlier_mask = (df['statut_ligne'] == 'ACTIF') & ((df[col] < lower) | (df[col] > upper))
    df.loc[outlier_mask, 'is_outlier'] = 1
    df[f'{col}_outlier'] = outlier_mask.astype(int)
    outlier_summary[col] = {'nb': int(outlier_mask.sum()), 'lower': lower, 'upper': upper}
    print(f'  {col:<22} : {outlier_mask.sum():>4} outliers  [min={lower:.2f}, max={upper:.2f}]')

total_outliers = df['is_outlier'].sum()
print(f'\n  Total lignes avec outlier : {total_outliers} ({100*total_outliers/len(df):.1f}%)')

In [ ]:
# Visualisation : Boxplots des 4 mesures (lignes ACTIVES uniquement)
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for i, col in enumerate(FEATURES):
    data = df_actif[col].dropna()
    axes[i].boxplot(data, patch_artist=True,
                    boxprops=dict(facecolor='#3498db', alpha=0.7),
                    medianprops=dict(color='#e74c3c', linewidth=2))
    axes[i].set_title(col.replace('_', '\n'), fontsize=10, fontweight='bold')
    axes[i].set_ylabel('dBm')
    axes[i].axhline(PHYSICAL_LIMITS[col][0], color='orange', linestyle='--', linewidth=1.5, label='Min physique')
    axes[i].axhline(PHYSICAL_LIMITS[col][1], color='orange', linestyle='--', linewidth=1.5, label='Max physique')
    axes[i].legend(fontsize=7)

plt.suptitle('Boxplots des mesures XPON (lignes ACTIVES)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Étape 3 — Correction de la divergence géographique

La colonne `gouvernorat` (issue de la jointure officielle `CO → GOUVERNORAT`) est la **référence**.  
La colonne `DESGOUV` (texte libre dans `XPON_QUALIF`) peut contenir des erreurs.

In [ ]:
if 'DESGOUV' in df.columns:
    divergence = df[
        df['gouvernorat'].notna() &
        (df['DESGOUV'].str.strip().str.upper() != df['gouvernorat'].str.strip().str.upper())
    ]
    if len(divergence) > 0:
        print(f'[!] {len(divergence)} ligne(s) avec divergence DESGOUV <> gouvernorat (CO) :')
        display(divergence[['id_mesure', 'DESGOUV', 'gouvernorat']])
        print('-> Correction : colonne gouvernorat (source officielle) conservee')
    else:
        print('[OK] Aucune divergence detectee')

    df.drop(columns=['DESGOUV'], inplace=True, errors='ignore')
    print('-> Colonne DESGOUV supprimee (remplacee par gouvernorat)')
else:
    print('[OK] Colonne DESGOUV absente, pas de correction necessaire')

---
## Étape 4 — Détection et suppression des doublons

In [ ]:
nb_avant = len(df)
doublons = df[df.duplicated(subset=['id_mesure'], keep=False)]

if len(doublons) > 0:
    print(f'[!] {len(doublons)} doublons detectes sur id_mesure :')
    display(doublons[['id_mesure', 'id_carte_olt', 'onu_id']].head(5))
    df = df.drop_duplicates(subset=['id_mesure'], keep='first')
    print(f'-> {nb_avant - len(df)} doublons supprimes')
    print(f'-> Lignes restantes : {len(df)}')
else:
    print('[OK] Aucun doublon sur id_mesure')

---
## Étape 5 — Validation finale

In [ ]:
checks = {
    'Aucune valeur nulle sur id_mesure'    : df['id_mesure'].isna().sum() == 0,
    'Aucune valeur nulle sur id_carte_olt' : df['id_carte_olt'].isna().sum() == 0,
    'Aucune valeur nulle sur id_geo'       : df['id_geo'].isna().sum() == 0,
    'Colonne statut_ligne complete'        : df['statut_ligne'].isna().sum() == 0,
    'Aucun doublon sur id_mesure'          : not df.duplicated(subset=['id_mesure']).any(),
    'Colonne is_outlier presente'          : 'is_outlier' in df.columns,
}

all_ok = True
for label, passed in checks.items():
    status = '[OK]    ' if passed else '[ECHEC] '
    if not passed:
        all_ok = False
    print(f'{status} {label}')

print(f'\nStatut global : {"PRET POUR POSTGRESQL" if all_ok else "VERIFIER LES ECHECS"}')

In [3]:
data1= pd.read_excel('olap_clean.xlsx')
data1.dropna(inplace=True)

data1 = data1.drop_duplicates()


In [6]:
data1.to_excel('dataset_olap.xlsx',index=False)

---
## Rapport de qualité final

In [ ]:
total          = len(df)
actif          = (df['statut_ligne'] == 'ACTIF').sum()
inactif        = (df['statut_ligne'] == 'INACTIF').sum()
data_manquante = (df['statut_ligne'] == 'DONNEE_MANQUANTE').sum()
outliers_total = df['is_outlier'].sum()
incoherences   = (df.get('nb_incoherences', pd.Series(dtype=int)) >= 1).sum()

rapport = pd.DataFrame({
    'Indicateur': [
        'Lignes initiales', 'Lignes apres nettoyage',
        'Lignes ACTIVES', 'Lignes INACTIVES',
        'Lignes DONNEE_MANQUANTE', 'Outliers physiques', 'Incoherences flags'
    ],
    'Nombre': [
        nb_initial, total,
        actif, inactif,
        data_manquante, outliers_total, incoherences
    ],
    'Pourcentage': [
        100, round(100*total/nb_initial,1),
        round(100*actif/total,1), round(100*inactif/total,1),
        round(100*data_manquante/total,1), round(100*outliers_total/total,1),
        round(100*incoherences/total,1)
    ]
})
display(rapport)

# Distribution par Gouvernorat
print('\nDistribution par Gouvernorat :')
print(df['gouvernorat'].value_counts().to_string())

# Distribution par Marque
print('\nDistribution par Marque OLT :')
if 'marque' in df.columns:
    print(df['marque'].value_counts().to_string())

In [ ]:
# Graphique : Résumé qualite
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Statuts
statut_data = df['statut_ligne'].value_counts()
colors_s = ['#2ecc71', '#e74c3c', '#f39c12']
axes[0].pie(statut_data.values, labels=statut_data.index, autopct='%1.1f%%',
            colors=colors_s[:len(statut_data)], startangle=90)
axes[0].set_title('Statuts des lignes', fontweight='bold')

# 2. Gouvernorats
gouv_data = df['gouvernorat'].value_counts()
gouv_data.plot(kind='barh', ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Mesures par Gouvernorat', fontweight='bold')
axes[1].set_xlabel('Nombre de mesures')

# 3. Marques
if 'marque' in df.columns:
    marque_data = df['marque'].value_counts()
    axes[2].pie(marque_data.values, labels=marque_data.index, autopct='%1.1f%%',
                colors=['#9b59b6', '#e67e22'], startangle=90)
    axes[2].set_title('Repartition par Marque OLT', fontweight='bold')

plt.suptitle('Rapport de Qualite — Dataset XPON Nettoye', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Export final — `olap_clean.xlsx`

In [ ]:
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    df.to_excel(writer,        sheet_name='FAIT_MESURE_OLT', index=False)
    dim_equip.to_excel(writer, sheet_name='DIM_EQUIPEMENT',  index=False)
    dim_geo.to_excel(writer,   sheet_name='DIM_GEOGRAPHIE',  index=False)
    dim_temps.to_excel(writer, sheet_name='DIM_TEMPS',       index=False)

for sheet in ['FAIT_MESURE_OLT', 'DIM_EQUIPEMENT', 'DIM_GEOGRAPHIE', 'DIM_TEMPS']:
    print(f'  Feuille [{sheet}] exportee')

print(f'\n[OK] Fichier exporte : {OUTPUT_FILE}')
print(f'[OK] {len(df)} lignes propres pret pour PostgreSQL')
print('\nProchaine etape -> Executer schema.sql dans PostgreSQL')